# Verifying Data Lineage in AIDP

**What this notebook does:** builds a tiny 3-hop Delta pipeline in the AIDP lakehouse, captures its
lineage from Spark's own analyzed logical plan, then **asserts the captured lineage matches the
pipeline we actually wrote** — table-level *and* column-level — and cross-checks it against two
independent sources of truth (the Delta transaction log, and the physical files on OCI Object Storage).

### Why it's built this way

AIDP **does** ship a lineage API — the **`DataLineage`** service (`DataLineageClient`, CLI group
`data-lineage`), on the data-plane host `datalake.{region}.oci.oraclecloud.com` at API version
**`/20260430`**. It shipped as `SemanticCatalog` in SDK v4.1.0 and was renamed in **v4.1.1
(2026-08-31)**, flagged breaking in the SDK changelog — pin any reference you write to the SDK
version you checked:

| Operation | Call |
|---|---|
| Fetch entity lineage | `POST /20260430/aiDataPlatforms/{id}/actions/fetchLineage` |
| Export lineage (CSV) | `POST /20260430/aiDataPlatforms/{id}/actions/exportLineage` |

It supports `level: ENTITY \| COLUMN`, `direction: UPSTREAM \| DOWNSTREAM \| BOTH`, `maxDepth`, node and
path filters, and returns `EntityLineage { nodes[], links[] }`. Both operations are labelled
**(Preview)** in the official CLI reference. `test_aidp_lineage_api.py` in this folder proves the
endpoint is live from real responses.

That API is the *platform's* claim about lineage. This notebook is the **independent oracle** you check
it against: lineage derived from the one thing that cannot disagree with what actually ran — the
Catalyst **analyzed logical plan** of each write, the same signal OpenLineage/Spline-style collectors
read. Two independent derivations that agree is what makes a lineage claim trustworthy; a platform
graph nobody cross-checks is just an assertion.

It also still works when the platform graph is empty. Observed 2026-08-16 on one DataLake in
us-ashburn-1 with SDK v4.2.1, `fetchLineage` rejected every candidate `anchorNode` for tables there
with `400 Invalid anchorNode` — the API is released, but no graph came back on that tenancy. Yours may
differ; the README says how to check.

### The three verification layers

| Layer | Source of truth | Proves |
|---|---|---|
| 1. Plan-derived graph | Catalyst analyzed plan | which tables/columns fed each write |
| 2. Delta history | `_delta_log` commit log | that the write happened, with row counts |
| 3. File provenance | `inputFiles()` + `DESCRIBE DETAIL` | bytes read live under the claimed table |

A run is only green if all three agree **and** the negative controls stay clean.

## 0 · Config
Everything is created inside one throwaway schema and dropped at the end.

In [ ]:
SCHEMA = "default.lin_demo"        # <catalog>.<schema> — change if you want a different landing spot

RAW_CUST, RAW_ORD = SCHEMA + ".raw_customers", SCHEMA + ".raw_orders"
STG               = SCHEMA + ".stg_orders"
MART              = SCHEMA + ".mart_customer_revenue"
DECOY             = SCHEMA + ".unrelated_table"     # negative control: must never appear in lineage

ALL_TABLES = [RAW_CUST, RAW_ORD, STG, MART, DECOY]

CHECKS = []   # (name, passed, detail) — filled in by the verification cells
def check(name, passed, detail=""):
    CHECKS.append((name, bool(passed), detail))
    print("%s  %s%s" % ("PASS" if passed else "FAIL", name, ("  — " + detail) if detail else ""))
    return bool(passed)

print("spark", spark.version, "| default format:", spark.conf.get("spark.sql.sources.default"))
print("target schema:", SCHEMA)

## 1 · The lineage extractor

Reads lineage out of the **analyzed logical plan**.

- **Table level** — `plan.collectLeaves()` gives every leaf relation; `catalogTable().qualifiedName()`
  turns each into a clean `catalog.schema.table`.
- **Column level** — every output expression's `references()` are `AttributeReference`s carrying an
  `exprId`. Mapping those `exprId`s back to the leaf relations' output attributes gives
  `output_column <- [(source_table, source_column), ...]`.

  Aliases and aggregates mint *fresh* `exprId`s (`SUM(o.amount) AS revenue` is a new attribute), which is
  exactly why we resolve through `references()` rather than comparing output ids to leaf ids directly.

In [ ]:
def _leaf_sources(analyzed):
    """-> ([table names], {exprId: (table, column)}) for every leaf relation in the plan."""
    colmap, tables = {}, []
    leaves = analyzed.collectLeaves()
    for i in range(leaves.size()):
        lf = leaves.apply(i)
        name = None
        try:
            ct = lf.catalogTable()
            if ct.isDefined():
                name = ct.get().qualifiedName()
        except Exception:
            pass
        if name is None:                       # non-catalog leaf: files, range(), inline literals
            name = "<%s>" % lf.getClass().getSimpleName()
        tables.append(name)
        try:
            out = lf.output()
            for j in range(out.size()):
                a = out.apply(j)
                colmap[a.exprId().id()] = (name, a.name())
        except Exception:
            pass
    return tables, colmap


def _output_exprs(analyzed):
    """Top-level output expressions: Project -> projectList, Aggregate -> aggregateExpressions."""
    # Dispatch on the node type, not hasattr(): py4j synthesises attributes on demand, so
    # hasattr() is True for every name on a JavaObject and the loop would always take its
    # first branch regardless of what the node actually is.
    node = analyzed.getClass().getSimpleName()
    accessor = {"Project": "projectList", "Aggregate": "aggregateExpressions"}.get(node)
    if accessor is None:
        return None, node
    try:
        return getattr(analyzed, accessor)(), node
    except Exception:
        return None, node


def lineage_of(df):
    """-> {tables: [...], columns: {out_col: [(tbl, col), ...]}, top: str}"""
    analyzed = df._jdf.queryExecution().analyzed()
    tables, colmap = _leaf_sources(analyzed)
    exprs, top = _output_exprs(analyzed)
    columns = {}
    if exprs is not None:
        for i in range(exprs.size()):
            e = exprs.apply(i)
            try:
                name = e.name()
            except Exception:
                continue          # non-named expression (rare at top level)
            refs, ups = e.references().toSeq(), []
            for k in range(refs.size()):
                up = colmap.get(refs.apply(k).exprId().id())
                if up and up not in ups:
                    ups.append(up)
            columns[name] = ups
    return {"tables": sorted(set(tables)), "columns": columns, "top": top}


LINEAGE = []      # the captured lineage graph

def write_tracked(target, sql, mode="overwrite"):
    """Run `sql`, capture its lineage from the plan, then write to `target` as Delta."""
    df  = spark.sql(sql)
    lin = lineage_of(df)                                  # capture BEFORE the write
    df.write.format("delta").mode(mode).saveAsTable(target)
    LINEAGE.append({"target": target, "sources": lin["tables"], "columns": lin["columns"]})
    print("  captured: %s\n         <- %s" % (target, ", ".join(lin["tables"]) or "(none)"))

print("extractor ready")

## 2 · Seed the raw tables
Four paid/cancelled orders across three customers, plus a decoy table that no pipeline reads.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS " + SCHEMA)
for t in ALL_TABLES:
    spark.sql("DROP TABLE IF EXISTS " + t)

spark.createDataFrame(
    [(1, "Acme", "US"), (2, "Globex", "DE"), (3, "Initech", "US")],
    "cust_id int, cust_name string, country string",
).write.format("delta").mode("overwrite").saveAsTable(RAW_CUST)

spark.createDataFrame(
    [(101, 1, 250.00, "PAID"), (102, 1, 125.50, "PAID"),
     (103, 2,  80.00, "CANCELLED"), (104, 3, 410.25, "PAID")],
    "order_id int, cust_id int, amount double, status string",
).write.format("delta").mode("overwrite").saveAsTable(RAW_ORD)

# never read by any hop — if it shows up in lineage, the extractor is over-reporting
spark.createDataFrame([(9, "noise")], "x int, y string") \
    .write.format("delta").mode("overwrite").saveAsTable(DECOY)

spark.sql("SELECT * FROM " + RAW_ORD + " ORDER BY order_id").show()

## 3 · Build the pipeline (lineage captured as it runs)

```
raw_orders ──filter status='PAID'──> stg_orders ──┐
                                                  ├─join + group by──> mart_customer_revenue
raw_customers ────────────────────────────────────┘

unrelated_table                    (decoy — no edges)
```

Note `mart` reads **`stg_orders`**, not `raw_orders`. `raw_orders` is only *transitively* upstream —
a correct extractor must report the direct edge only. That distinction is asserted in §5.

In [ ]:
print("-- hop 1: raw_orders -> stg_orders --")
write_tracked(STG, "SELECT order_id, cust_id, amount FROM %s WHERE status = 'PAID'" % RAW_ORD)

print("\n-- hop 2: stg_orders + raw_customers -> mart (join + aggregate) --")
write_tracked(MART, """
    SELECT c.cust_id, c.cust_name, c.country,
           SUM(o.amount)      AS revenue,
           COUNT(o.order_id)  AS n_orders
    FROM {stg} o
    JOIN {cust} c ON o.cust_id = c.cust_id
    GROUP BY c.cust_id, c.cust_name, c.country
""".format(stg=STG, cust=RAW_CUST))

print()
spark.sql("SELECT * FROM %s ORDER BY cust_id" % MART).show()

## 4 · Verify table-level lineage against the expected DAG
The real test: compare the captured graph to the DAG we intended, as **sets** — so a missing edge *and*a spurious edge both fail.

In [ ]:
EXPECTED = {
    STG:  {RAW_ORD},
    MART: {STG, RAW_CUST},
}

captured = {r["target"]: set(r["sources"]) for r in LINEAGE}

check("every expected target was captured",
      set(EXPECTED) == set(captured),
      "expected %d targets, captured %d" % (len(EXPECTED), len(captured)))

for target, want in EXPECTED.items():
    got = captured.get(target, set())
    short = lambda s: sorted(x.split(".")[-1] for x in s)
    ok = got == want
    detail = "sources = %s" % short(got) if ok else "want %s, got %s" % (short(want), short(got))
    check("lineage of %-22s" % target.split(".")[-1], ok, detail)

## 5 · Negative controls — precision, not just recall

Recall is easy; a extractor that reported *every* table would pass §4's "no missing edges" half. These
two checks are what make the result meaningful:

1. The **decoy** table appears in no edge at all.
2. `mart` does **not** claim a direct edge to `raw_orders` — that dependency is real but *transitive*.

In [ ]:
all_sources = set()
for r in LINEAGE:
    all_sources |= set(r["sources"])

check("decoy table absent from all lineage", DECOY not in all_sources,
      "%s never read" % DECOY.split(".")[-1])

check("no direct mart -> raw_orders edge (transitive only)",
      RAW_ORD not in captured[MART],
      "mart's direct sources: %s" % sorted(x.split(".")[-1] for x in captured[MART]))

check("no unresolved/non-catalog leaves leaked in",
      not any(s.startswith("<") for s in all_sources),
      "all %d sources resolved to catalog tables" % len(all_sources))

## 6 · Verify column-level lineage
Table-level lineage says *mart depends on stg_orders*. Column-level says **`revenue` comes from`stg_orders.amount`** — the claim that actually matters for impact analysis.

In [ ]:
EXPECTED_COLS = {
    (STG,  "amount"):    [(RAW_ORD,  "amount")],
    (STG,  "order_id"):  [(RAW_ORD,  "order_id")],
    (MART, "revenue"):   [(STG,      "amount")],
    (MART, "n_orders"):  [(STG,      "order_id")],
    (MART, "cust_name"): [(RAW_CUST, "cust_name")],
}

colmaps = {r["target"]: r["columns"] for r in LINEAGE}

for (tbl, col), want in EXPECTED_COLS.items():
    got = [tuple(x) for x in colmaps.get(tbl, {}).get(col, [])]
    fmt = lambda ps: ", ".join("%s.%s" % (t.split(".")[-1], c) for t, c in ps) or "(none)"
    check("column %-16s" % (tbl.split(".")[-1] + "." + col),
          got == want,
          "<- %s" % fmt(got) if got == want else "want %s, got %s" % (fmt(want), fmt(got)))

print("\nFull column map for %s:" % MART.split(".")[-1])
for col, ups in colmaps[MART].items():
    # "(no source columns resolved)" rather than "(literal)": an empty list means either a
    # genuine constant OR an attribute minted by an intermediate Project, which this
    # extractor cannot resolve. The two are indistinguishable here -- see Scope in README.
    print("   %-12s <- %s" % (col, ", ".join("%s.%s" % (t.split('.')[-1], c) for t, c in ups)
                              or "(no source columns resolved)"))

## 7 · Cross-check 1 — the Delta transaction log

Independent of Catalyst: every write left a commit in `_delta_log`. This confirms the writes are real
and that row counts match what the transformation logic implies 
(3 PAID orders -> 3 staged rows;
2 customers with paid orders -> 2 mart rows).

Note what the commit log **does not** contain: any reference to the *input* tables. `operationParameters`
for a CTAS carries partitioning and properties, not sources. That gap is precisely why §1 exists —
Delta history gives temporal provenance, not a lineage graph.

In [ ]:
EXPECTED_ROWS = {STG: 3, MART: 2}

for t in (STG, MART):
    h = spark.sql("DESCRIBE HISTORY %s" % t).orderBy("version").collect()
    last = h[-1]
    metrics = last["operationMetrics"] or {}
    rows = int(metrics.get("numOutputRows", -1))
    print("%-24s v%-2s %-34s rows=%-4s engine=%s" % (
        t.split(".")[-1], last["version"], last["operation"], rows, (last["engineInfo"] or "?")))
    check("delta commit exists for %-14s" % t.split(".")[-1], len(h) >= 1,
          "%d commit(s)" % len(h))
    check("row count matches transform for %-8s" % t.split(".")[-1],
          rows == EXPECTED_ROWS[t], "%d rows (expected %d)" % (rows, EXPECTED_ROWS[t]))

print("\noperationParameters of the mart commit (note: no source tables listed):")
print("  ", spark.sql("DESCRIBE HISTORY %s" % MART).collect()[0]["operationParameters"])

## 8 · Cross-check 2 — physical file provenance
The strongest check: when Spark reads a table, the **actual file paths** it opens must live under thattable's registered storage location. This ties the logical lineage claim to bytes on OCI Object Storage.

In [ ]:
for t in (RAW_ORD, STG, MART):
    loc   = spark.sql("DESCRIBE DETAIL %s" % t).collect()[0]["location"]
    files = spark.sql("SELECT * FROM %s" % t).inputFiles()
    under = all(f.startswith(loc) for f in files)
    check("files of %-22s under its location" % t.split(".")[-1],
          bool(files) and under, "%d file(s)" % len(files))
    # The check above uses the full location; only the path is printed. The authority
    # (oci://<bucket>@<namespace>) identifies your tenancy and must not reach a
    # committed notebook output.
    print("     loc: .../%s" % loc.split("/", 3)[-1])

## 9 · Traverse the graph — multi-hop lineage
With a verified edge list, the questions people actually ask become graph walks: *what feeds thistable (transitively)?* and *what breaks if I change this one?*

In [ ]:
upstream_map = {r["target"]: list(r["sources"]) for r in LINEAGE}
downstream_map = {}
for tgt, srcs in upstream_map.items():
    for s in srcs:
        downstream_map.setdefault(s, []).append(tgt)

def walk(node, mapping, seen=None, depth=0):
    """Transitive closure with cycle guard."""
    seen = seen or set()
    if node in seen:
        return
    seen.add(node)
    for nxt in sorted(mapping.get(node, [])):
        yield depth, nxt
        for d, n in walk(nxt, mapping, seen, depth + 1):
            yield d, n

print("UPSTREAM of %s (what it is built from):" % MART.split(".")[-1])
for d, n in walk(MART, upstream_map):
    print("   " + "   " * d + "└─ " + n.split(".")[-1])

print("\nDOWNSTREAM of %s (blast radius of a change):" % RAW_ORD.split(".")[-1])
for d, n in walk(RAW_ORD, downstream_map):
    print("   " + "   " * d + "└─ " + n.split(".")[-1])

roots = [n for _, n in walk(MART, upstream_map) if n not in upstream_map]
check("transitive roots of mart are the raw tables",
      set(roots) == {RAW_ORD, RAW_CUST},
      "roots = %s" % sorted(r.split(".")[-1] for r in roots))

## 10 · Scorecard

In [ ]:
passed = sum(1 for _, ok, _ in CHECKS if ok)
total  = len(CHECKS)

print("=" * 68)
for name, ok, detail in CHECKS:
    print(" %-5s %s" % ("PASS" if ok else "FAIL", name.strip()))
print("=" * 68)
print(" %d/%d checks passed" % (passed, total))
print("=" * 68)

if passed == total:
    print("\nLINEAGE VERIFIED — plan-derived graph, Delta commit log, and physical")
    print("file layout all agree, and both negative controls stayed clean.")
else:
    print("\n%d CHECK(S) FAILED — see above." % (total - passed))
    for name, ok, detail in CHECKS:
        if not ok:
            print("   - %s: %s" % (name.strip(), detail))

## What this does and does not prove

*Run 2026-08-16 on Spark 3.5.0 / Delta 3.2.0-oci-1.0.0: 19/19.*

**Proves** — for work that runs through `write_tracked` whose top plan node is a `Project` or an
`Aggregate` **and every attribute that node references is emitted directly by a leaf relation**,
lineage can be extracted from Spark on AIDP at both table and column granularity, verified against an
expected DAG with negative controls, and corroborated by two independent sources (Delta commit log,
physical file paths).

That second condition is the real limit, and it is easy to miss. `colmap` is built only from
`collectLeaves()`, so an attribute minted by an *intermediate* `Project` — an alias computed in a
`FROM`-subquery, or an earlier `.withColumn()` / `.select()` — is absent from the map no matter what
the top node is, and the column resolves to nothing. `SELECT * FROM (SELECT amount*2 AS amt FROM
raw_orders) s` has a `Project` on top and still yields no lineage for `amt`.

The 19/19 result is scoped accordingly. Window functions, `Project` over `Project`, `Project` over
`Aggregate`, `LATERAL VIEW explode`, `UNION` branches, CTEs and subquery sources are **not** handled
correctly, and a top node of `Sort`/`Distinct`/`Filter`/`WithCTE` yields an empty column map without
warning. See "Scope" in `README.md` for the full table. This is a worked demonstration of the
technique, not a general-purpose extractor.

**Does not prove** — that AIDP captures lineage *platform-wide on its own*. It does not:

- Lineage is captured only for writes routed through `write_tracked`. A plain
  `df.write.saveAsTable(...)` elsewhere in the workspace is invisible to this graph, and `LINEAGE`
  lives in kernel memory — it dies with the session.
- Column lineage resolves the **top** plan node (`Project` / `Aggregate`). Columns consumed only in
  `WHERE` / `JOIN ON` predicates (e.g. `raw_orders.status`, which the filter reads) are dependencies
  but are not top-level outputs, so they do not appear in the column map. Deeper coverage means
  walking the whole plan tree, not just its root.
- Nothing is published to AIDP's own `DataLineage` graph. That API is read-only for
  lineage (`fetchLineage` / `exportLineage`) — it exposes no write/ingest operation, so a client
  cannot push this graph into it. Population is the platform's job.

**To make it durable**, the two production-grade options:

1. **`QueryExecutionListener`** — register a listener on `spark.listenerManager` so *every* write is
   captured automatically, with no call-site changes. Needs a JVM-side listener (Scala/Java JAR, or a
   py4j callback) rather than the explicit wrapper used here.
2. **OpenLineage Spark listener** — install `io.openlineage:openlineage-spark` as a cluster library and
   point `spark.openlineage.transport.*` at a collector (Marquez, or any OpenLineage-compatible
   backend). Set `spark.extraListeners=io.openlineage.spark.agent.OpenLineageSparkListener`. This emits
   the same plan-derived lineage this notebook computes by hand, to durable external storage.

Either way the *signal* is what §1 demonstrates: the analyzed logical plan. This notebook verifies that
signal is correct and complete on AIDP before you wire it to a collector.

## 11 · Cleanup

In [ ]:
for t in ALL_TABLES:
    spark.sql("DROP TABLE IF EXISTS " + t)
spark.sql("DROP SCHEMA IF EXISTS %s CASCADE" % SCHEMA)
print("dropped %d tables and schema %s" % (len(ALL_TABLES), SCHEMA))